In [2]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import copy
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de treinamento: {device}")


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) 
])

train_path = './cifake/train'
train_dataset = ImageFolder(root=train_path, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

val_path = './cifake/test'
val_dataset = ImageFolder(root=val_path, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


dataloaders = {
    'train': train_loader,
    'val': val_loader
}


class MobileNetV3Detector(nn.Module):
    def __init__(self):
        super(MobileNetV3Detector, self).__init__()
        
        
        self.base_model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
        
        
        num_features = self.base_model.classifier[0].in_features 
        
        
        self.base_model.classifier = nn.Identity() 
        
        
        self.custom_classifier = nn.Sequential(
            nn.BatchNorm1d(num_features),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),    
            nn.Linear(256, 64),          
            nn.ReLU(),
            nn.Linear(64, 1)             
        )

    def forward(self, x):
        x = self.base_model(x)
        x = self.custom_classifier(x)
        return x

model = MobileNetV3Detector().to(device)
criterion = nn.BCEWithLogitsLoss()

def train_with_early_stopping(model, optimizer, criterion, dataloaders, max_epochs=20, patience=3):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        print(f'\nEpoch {epoch+1}/{max_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            correct = 0 
            total = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.unsqueeze(1).float().to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item()
                
                with torch.no_grad():
                    predicted = torch.round(torch.sigmoid(outputs.data)) 
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

            epoch_loss = running_loss / len(dataloaders[phase])
            epoch_acc = correct / total

            print(f'{phase.capitalize()} -> Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}')

            
            if phase == 'val':
                if epoch_loss < best_loss:
                    best_loss = epoch_loss
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
                    print(f'Sem melhoria por {epochs_no_improve} época(s).')

        if epochs_no_improve >= patience:
            print('\n>>> Early Stopping ativado! Interrompendo para evitar overfitting. <<<')
            break

    
    model.load_state_dict(best_model_wts)
    return model

def extract_metrics(model, dataloader, device, dataset_name):
    model.eval() 
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds = torch.round(torch.sigmoid(outputs))
            
            all_preds.extend(preds.squeeze().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    print(f"\n{'-'*40}")
    print(f"RELATÓRIO: {dataset_name.upper()}")
    print(f"{'-'*40}")
    
    report = classification_report(all_labels, all_preds, target_names=['Fake', 'Real'], digits=4)
    print(report)


print("INICIANDO FASE 1 (Base Congelada)")

for param in model.base_model.parameters():
    param.requires_grad = False
    
for param in model.custom_classifier.parameters():
    param.requires_grad = True


optimizer_fc = optim.Adamax(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

model = train_with_early_stopping(model, optimizer_fc, criterion, dataloaders, max_epochs=20, patience=1)

extract_metrics(model, val_loader, device, "Teste")
extract_metrics(model, train_loader, device, "Treinamento")

print("INICIANDO FASE 2 (Fine Tuning)")

for param in model.base_model.parameters():
    param.requires_grad = True

optimizer_full = optim.Adamax(model.parameters(), lr=0.001)

model = train_with_early_stopping(model, optimizer_full, criterion, dataloaders, max_epochs=20, patience=1)

extract_metrics(model, val_loader, device, "Teste")
extract_metrics(model, train_loader, device, "Treinamento")


Dispositivo de treinamento: cuda
INICIANDO FASE 1 (Base Congelada)

Epoch 1/20
----------
Train -> Loss: 0.5780, Accuracy: 0.6967
Val -> Loss: 0.5597, Accuracy: 0.7147

Epoch 2/20
----------
Train -> Loss: 0.5622, Accuracy: 0.7108
Val -> Loss: 0.5510, Accuracy: 0.7183

Epoch 3/20
----------
Train -> Loss: 0.5574, Accuracy: 0.7139
Val -> Loss: 0.5455, Accuracy: 0.7222

Epoch 4/20
----------
Train -> Loss: 0.5525, Accuracy: 0.7159
Val -> Loss: 0.5441, Accuracy: 0.7243

Epoch 5/20
----------
Train -> Loss: 0.5488, Accuracy: 0.7199
Val -> Loss: 0.5415, Accuracy: 0.7280

Epoch 6/20
----------
Train -> Loss: 0.5437, Accuracy: 0.7248
Val -> Loss: 0.5356, Accuracy: 0.7283

Epoch 7/20
----------
Train -> Loss: 0.5411, Accuracy: 0.7271
Val -> Loss: 0.5380, Accuracy: 0.7307
Sem melhoria por 1 época(s).

>>> Early Stopping ativado! Interrompendo para evitar overfitting. <<<

----------------------------------------
RELATÓRIO: TESTE
----------------------------------------
              precision  